# EMAGE — audio largo en Google Colab

Objetivo: subir un audio de hasta ~3 minutos y generar movimiento con **EMAGE**.

No usa ZeroGPU ni renderizadores. Procesa por bloques de 8 s con 1 s de solapamiento y une el movimiento al final.

Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**.


In [ ]:
import sys, subprocess, importlib
print('Python:', sys.version)
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['rm','-rf','/content/PantoMatrix'], check=True)
subprocess.run(['git','clone','--depth','1','https://github.com/PantoMatrix/PantoMatrix.git','/content/PantoMatrix'], check=True)
needed={'librosa':'librosa','soundfile':'soundfile','einops':'einops','omegaconf':'omegaconf','huggingface_hub':'huggingface_hub','tqdm':'tqdm'}
missing=[]
for mod,pkg in needed.items():
    try: importlib.import_module(mod)
    except Exception: missing.append(pkg)
if missing:
    print('Instalando paquetes faltantes:', missing)
    subprocess.run([sys.executable,'-m','pip','install','-q',*missing], check=True)
import torch, numpy, librosa, soundfile, transformers, huggingface_hub, einops
assert torch.cuda.is_available(), 'No hay GPU activa. Elegí GPU en el entorno de ejecución.'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:',torch.__version__,'| transformers:',transformers.__version__,'| librosa:',librosa.__version__)
print('✅ Preparación terminada de verdad')


In [ ]:
from google.colab import files
from pathlib import Path
import shutil
folder=Path('/content/input_audio')
shutil.rmtree(folder,ignore_errors=True); folder.mkdir()
uploaded=files.upload()
name,data=next(iter(uploaded.items()))
audio_path=folder/name; audio_path.write_bytes(data)
print('✅ Audio cargado:',audio_path)


In [ ]:
import sys, math, json, numpy as np, torch, librosa
import torch.nn.functional as F
from pathlib import Path
sys.path.insert(0,'/content/PantoMatrix')
from models.emage_audio import EmageAudioModel, EmageVQVAEConv, EmageVAEConv, EmageVQModel

# PantoMatrix fue escrito para Transformers 4.x. Colab actual usa Transformers 5.x,
# que espera este atributo en modelos custom antes de from_pretrained().
for cls in (EmageVQVAEConv, EmageVAEConv, EmageAudioModel):
    if not hasattr(cls,'all_tied_weights_keys'):
        cls.all_tied_weights_keys={}

device=torch.device('cuda')
print('Cargando EMAGE...')
face=EmageVQVAEConv.from_pretrained('H-Liu1997/emage_audio',subfolder='emage_vq/face').to(device)
upper=EmageVQVAEConv.from_pretrained('H-Liu1997/emage_audio',subfolder='emage_vq/upper').to(device)
lower=EmageVQVAEConv.from_pretrained('H-Liu1997/emage_audio',subfolder='emage_vq/lower').to(device)
hands=EmageVQVAEConv.from_pretrained('H-Liu1997/emage_audio',subfolder='emage_vq/hands').to(device)
global_ae=EmageVAEConv.from_pretrained('H-Liu1997/emage_audio',subfolder='emage_vq/global').to(device)
motion_vq=EmageVQModel(face_model=face,upper_model=upper,lower_model=lower,hands_model=hands,global_model=global_ae).to(device).eval()
model=EmageAudioModel.from_pretrained('H-Liu1997/emage_audio').to(device).eval()
sr=int(model.cfg.audio_sr); fps=int(model.cfg.pose_fps)
print(f'✅ EMAGE cargado · {sr} Hz · {fps} fps')


In [ ]:
CHUNK_SEC=8.0
OVERLAP_SEC=1.0
audio,_=librosa.load(str(audio_path),sr=sr,mono=True)
duration=len(audio)/sr
print(f'Duración total: {duration:.2f}s')
if duration>180: print('⚠️ Supera 3 minutos; se intentará igual.')
def infer_chunk(wave):
    x=torch.from_numpy(wave.astype(np.float32)).to(device).unsqueeze(0)
    speaker=torch.zeros(1,1,dtype=torch.long,device=device)
    ref_trans=torch.zeros(1,3,device=device)
    with torch.no_grad():
        z=model.inference(x,speaker,motion_vq,masked_motion=None,mask=None)
        out=motion_vq.decode(face_latent=z['rec_face'] if model.cfg.lf>0 and model.cfg.cf==0 else None,upper_latent=z['rec_upper'] if model.cfg.lu>0 and model.cfg.cu==0 else None,lower_latent=z['rec_lower'] if model.cfg.ll>0 and model.cfg.cl==0 else None,hands_latent=z['rec_hands'] if model.cfg.lh>0 and model.cfg.ch==0 else None,face_index=torch.max(F.log_softmax(z['cls_face'],dim=2),dim=2)[1] if model.cfg.cf>0 else None,upper_index=torch.max(F.log_softmax(z['cls_upper'],dim=2),dim=2)[1] if model.cfg.cu>0 else None,lower_index=torch.max(F.log_softmax(z['cls_lower'],dim=2),dim=2)[1] if model.cfg.cl>0 else None,hands_index=torch.max(F.log_softmax(z['cls_hands'],dim=2),dim=2)[1] if model.cfg.ch>0 else None,get_global_motion=True,ref_trans=ref_trans)
    return out['motion_axis_angle'][0].detach().cpu().numpy()
chunk_samples=int(CHUNK_SEC*sr); step=int((CHUNK_SEC-OVERLAP_SEC)*sr)
parts=[]; start=0; idx=0
while start<len(audio):
    end=min(len(audio),start+chunk_samples); wave=audio[start:end]; true_samples=len(wave)
    if true_samples<chunk_samples: wave=np.pad(wave,(0,chunk_samples-true_samples))
    print(f'[{idx+1}] {start/sr:.1f}–{end/sr:.1f}s')
    poses=infer_chunk(wave)
    parts.append(poses[:max(1,round((true_samples/sr)*fps))])
    if end>=len(audio): break
    start+=step; idx+=1
print('✅ Chunks generados:',len(parts))


In [ ]:
def aa_to_quat(v):
    x,y,z=map(float,v); a=math.sqrt(x*x+y*y+z*z)
    if a<1e-8:return np.array([0.,0.,0.,1.])
    s=math.sin(a/2)/a; return np.array([x*s,y*s,z*s,math.cos(a/2)],dtype=np.float64)
def quat_to_aa(q):
    q=np.asarray(q,dtype=np.float64); q=q/np.linalg.norm(q); w=np.clip(q[3],-1,1); a=2*math.acos(w); s=math.sqrt(max(1e-12,1-w*w))
    return q[:3]*2 if s<1e-6 else q[:3]/s*a
def slerp(a,b,t):
    a=a/np.linalg.norm(a); b=b/np.linalg.norm(b); dot=float(np.dot(a,b))
    if dot<0:b=-b;dot=-dot
    if dot>0.9995:
        q=a+(b-a)*t; return q/np.linalg.norm(q)
    th=math.acos(np.clip(dot,-1,1)); sn=math.sin(th); return (math.sin((1-t)*th)/sn)*a+(math.sin(t*th)/sn)*b
overlap_frames=round(OVERLAP_SEC*fps); merged=parts[0].copy()
for part in parts[1:]:
    n=min(overlap_frames,len(merged),len(part)); J=merged.shape[1]//3
    for f in range(n):
        t=(f+1)/(n+1)
        for j in range(J):
            a=aa_to_quat(merged[-n+f,j*3:j*3+3]); b=aa_to_quat(part[f,j*3:j*3+3]); merged[-n+f,j*3:j*3+3]=quat_to_aa(slerp(a,b,t))
    merged=np.concatenate([merged,part[n:]],axis=0)
merged=merged[:round(duration*fps)]
SMPLX={'hips':0,'leftUpperLeg':1,'rightUpperLeg':2,'spine':3,'leftLowerLeg':4,'rightLowerLeg':5,'chest':9,'leftFoot':10,'rightFoot':11,'neck':12,'leftShoulder':13,'rightShoulder':14,'head':15,'leftUpperArm':16,'rightUpperArm':17,'leftForeArm':18,'rightForeArm':19,'leftHand':20,'rightHand':21}
frames=[]
for row in merged:
    joints={}
    for name,j in SMPLX.items():
        joints[name]={'rotation':[0,0,0,1] if name=='hips' else aa_to_quat(row[j*3:j*3+3]).tolist()}
    frames.append({'root':{'position':[0,0,0]},'joints':joints})
result={'model':'pantomatrix','generator':'EMAGE via Google Colab','fps':fps,'source_duration_sec':duration,'frames':frames}
out=Path('/content/emage-motion.json'); out.write_text(json.dumps(result))
print(f'✅ LISTO · {len(frames)} frames · {len(frames)/fps:.2f}s')
files.download(str(out))
